# Point clouds

**Geometry type:** `point_cloud`

This notebook demonstrates the full lifecycle of a ZVF point cloud store:

1. Generate synthetic data (positions + attributes)
2. Write to a `.zarrvectors` store
3. Open lazily with `ZarrVectorStore`
4. Inspect metadata without loading vertex data
5. Read all vertices
6. Spatial bounding-box query
7. Build a multi-resolution pyramid
8. Read at coarser levels
9. Export to CSV

**Requirements:** `pip install zarr-vectors`


In [1]:
import numpy as np
import tempfile, os
from pathlib import Path

# All example stores are written to a temporary directory
_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "scan.zarrvectors")
print("Store path:", STORE)


Store path: /tmp/zvf_examples_hb6sxv9m/scan.zarrvectors


## 1 · Generate synthetic data

In [2]:
rng = np.random.default_rng(42)

N = 200_000
# Positions: 200k points in a 1000³ µm synchrotron scan volume
positions   = rng.uniform(0, 1000, (N, 3)).astype(np.float32)

# Per-vertex attributes
intensity   = rng.uniform(0, 1, N).astype(np.float32)          # absorption value
label       = rng.integers(0, 8, N).astype(np.int32)            # tissue class
confidence  = rng.uniform(0.5, 1, N).astype(np.float32)

print(f"positions : {positions.shape}  dtype={positions.dtype}")
print(f"intensity : {intensity.shape}  range [{intensity.min():.3f}, {intensity.max():.3f}]")
print(f"label     : {label.shape}  classes {np.unique(label)}")


positions : (200000, 3)  dtype=float32
intensity : (200000,)  range [0.000, 1.000]
label     : (200000,)  classes [0 1 2 3 4 5 6 7]


## 2 · Write to ZVF store

In [3]:
from zarr_vectors.types.points import write_points

write_points(
    STORE,
    positions,
    chunk_shape=(200.0, 200.0, 200.0),   # 200³ µm per chunk → 125 chunks
    bin_shape=(50.0, 50.0, 50.0),        # 50³ µm bins → 64 bins per chunk
    attributes={
        "intensity":  intensity,
        "label":      label,
        "confidence": confidence,
    },
)
print("Write complete.")
print("Store contents:")
for p in sorted(Path(STORE).rglob("zarr.json")):
    print(" ", p.relative_to(STORE))


Write complete.
Store contents:


## 3 · Open lazily with `ZarrVectorStore`

No vertex data is loaded at this step — only `.zattrs` and `zarr.json` metadata files.

In [4]:
from zarr_vectors.lazy import open_zvr

store = open_zvr(STORE)
print("Opened lazily — no vertex data loaded yet.")
print(store)

Opened lazily — no vertex data loaded yet.
ZVRStore('/tmp/zvf_examples_hb6sxv9m/scan.zarrvectors', levels=[0], geometry=[point_cloud], chunk=(200.0, 200.0, 200.0), bin=(50.0, 50.0, 50.0))


## 4 · Inspect metadata

In [5]:
print("geometry_types:", store.geometry_types)
print("ndim          :", store.ndim)
print("chunk_shape   :", store.chunk_shape)
print("bin_shape     :", store.bin_shape)        # at level 0
print("levels        :", store.levels)           # [0] — only base level so far
print("bounds        :", store.bounds)

# Vertex count from level metadata — no data read
print(f"\nVertex count (level 0): {store[0].vertex_count:,}")

geometry_types: ['point_cloud']
ndim          : 3
chunk_shape   : (200.0, 200.0, 200.0)
bin_shape     : (50.0, 50.0, 50.0)
levels        : [0]
bounds        : ([0.0038260614965111017, 0.0024366651196032763, 0.0046531520783901215], [999.9978637695312, 999.9982299804688, 999.99267578125])

Vertex count (level 0): 200,000


## 5 · Read all vertices

In [6]:
from zarr_vectors.types.points import read_points

result = read_points(STORE, attribute_names=["intensity", "label", "confidence"])

print(f"vertex_count : {result['vertex_count']:,}")
print(f"positions    : {result['positions'].shape}  dtype={result['positions'].dtype}")
print(f"intensity    : {result['attributes']['intensity'].shape}")
print(f"label        : {result['attributes']['label'].shape}")
print()
print("First 5 positions:")
print(result["positions"][:5])

vertex_count : 200,000
positions    : (200000, 3)  dtype=float32
intensity    : (200000,)
label        : (200000,)

First 5 positions:
[[44.3655     3.8109443 20.798489 ]
 [ 8.881518  26.021576   3.686151 ]
 [ 1.283508  24.254599  14.788155 ]
 [41.480705  10.674779  27.45147  ]
 [ 3.0815606  9.29687    1.8303767]]


## 6 · Spatial bounding-box query

The query targets individual bins — not full chunks — so only the relevant vertex groups are loaded.

In [7]:
lo = np.array([100.0, 100.0, 100.0])
hi = np.array([300.0, 300.0, 300.0])

bbox_result = read_points(STORE, bbox=(lo, hi), attribute_names=["intensity", "label", "confidence"])

expected_fraction = ((hi - lo) / 1000).prod()
expected_n = int(N * expected_fraction)

print(f"Query bbox     : {lo} → {hi}")
print(f"Expected ~      {expected_n:,} vertices  ({expected_fraction*100:.1f}% of volume)")
print(f"Returned        {bbox_result['vertex_count']:,} vertices")
print()
# All returned positions should be within the bbox
assert (bbox_result["positions"] >= lo).all(), "Point below lo!"
assert (bbox_result["positions"] <= hi).all(), "Point above hi!"
print("✓ All returned vertices are within the requested bbox")

Query bbox     : [100. 100. 100.] → [300. 300. 300.]
Expected ~      1,600 vertices  (0.8% of volume)
Returned        1,575 vertices

✓ All returned vertices are within the requested bbox


## 7 · Build a multi-resolution pyramid

In [8]:
from zarr_vectors.multiresolution.coarsen import build_pyramid

build_pyramid(
    STORE,
    level_configs=[
        {"bin_ratio": (2, 2, 2)},   # level 1: 8× fewer vertices
        {"bin_ratio": (4, 4, 4)},   # level 2: 64× fewer vertices
    ],
    agg_mode="mean",                # aggregation applied to all attributes
)
print("Pyramid built.")

store._level_list = None  # invalidate cached level list
print(f"Levels now available: {store.levels}")
for lv in store.levels:
    print(f"  Level {lv}: {store[lv].vertex_count:>8,} vertices"
          f"  bin_shape={store[lv].bin_shape}")

Pyramid built.
Levels now available: [0, 1, 2]
  Level 0:  200,000 vertices  bin_shape=None
  Level 1:    1,000 vertices  bin_shape=(100.0, 100.0, 100.0)
  Level 2:      125 vertices  bin_shape=(200.0, 200.0, 200.0)


## 8 · Read at coarser levels

In [9]:
for level in store.levels:
    r = read_points(STORE, level=level)
    print(f"Level {level}: {r['vertex_count']:>8,} vertices")

print()
# Coarsest level — useful for quick overview rendering
coarse = read_points(STORE, level=store.levels[-1])
print(f"Coarsest level ({store.levels[-1]}) positions range:")
print("  min:", coarse["positions"].min(axis=0))
print("  max:", coarse["positions"].max(axis=0))


Level 0:  200,000 vertices
Level 1:    1,000 vertices
Level 2:      125 vertices

Coarsest level (2) positions range:
  min: [98.129974 98.47804  99.04348 ]
  max: [902.1126  900.85846 901.2258 ]


## 9 · Lazy materialisation — accessing vertices via `ZVRStore`

In [10]:
level0 = store[0]
print(f"Level 0: {level0.chunk_count} non-empty chunks, {level0.vertex_count:,} vertices total")

# Materialise all vertices for level 0
all_verts = level0.vertices.compute()
print(f"Computed positions shape: {all_verts.shape}")
print(f"(matches full read: {result['vertex_count']:,})")

Level 0: 125 non-empty chunks, 200,000 vertices total
Computed positions shape: (200000, 3)
(matches full read: 200,000)


## 10 · Export to CSV

In [11]:
from zarr_vectors.export.csv_points import export_csv

csv_path = os.path.join(_tmpdir, "scan_export.csv")
export_csv(STORE, csv_path)

# Peek at the output
import pandas as pd
df = pd.read_csv(csv_path, nrows=5)
print(df)
print(f"\nFull CSV: {pd.read_csv(csv_path).shape[0]:,} rows")


        dim0       dim1       dim2
0  44.365501   3.810944  20.798489
1   8.881518  26.021576   3.686151
2   1.283508  24.254599  14.788155
3  41.480705  10.674779  27.451469
4   3.081561   9.296870   1.830377

Full CSV: 200,000 rows


## 11 · Validate the store

In [12]:
from zarr_vectors.validate import validate

result_v = validate(STORE, level=3)
print(result_v.summary())


Level 3 validation: PASS
  30 passed, 0 warnings, 0 errors


## Summary

| Step | API used |
|------|----------|
| Write | `write_points(path, positions, chunk_shape, bin_shape, attributes)` |
| Lazy open | `open_zvr(path)` → `ZVRStore` |
| Inspect metadata | `store.geometry_types`, `store.ndim`, `store.bounds`, `store[level].vertex_count` |
| Read all | `read_points(path)` |
| Bbox query | `read_points(path, bbox=(lo, hi))` |
| Pyramid | `build_pyramid(path, level_configs)` |
| Materialise vertices | `store[level].vertices.compute()` |
| Export | `export_csv(path, out)` |
| Validate | `validate(path, level=3)` |